In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

prompt = "I like cats"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("=== PREFILL ===")

with torch.no_grad():
    outputs = model(**inputs, use_cache=True)

past = outputs.past_key_values

for i, (k, v) in enumerate(past[:1]):
    print("Layer 0 K:", k.shape)
    print("Layer 0 V:", v.shape)

print("\n=== DECODE ===")

generated = inputs["input_ids"]

for step in range(3):
    with torch.no_grad():
        outputs = model(
            input_ids=generated[:, -1:],   # only last token
            past_key_values=past,
            use_cache=True
        )

    logits = outputs.logits
    past = outputs.past_key_values

    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    generated = torch.cat([generated, next_token], dim=1)

    print(f"Step {step+1} shape:", generated.shape)

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]